In [1]:
# colab_03_tahmin_uret.py  (v3 - val metrikleri: belirsizlik kapisi sizintisiz)
# Agent'in tuketecegi TEK dosya: 1115 magaza x 28 gun x 3 senaryo
# Origin 2015-07-03 -> 2015-07-04..07-31 (test penceresi, sizintisiz)
# Cikti: model/tahminler_tum_magazalar.parquet + model/tahmin_meta.json

import os, json
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [2]:
# ------------------------------------------------------------------
# 1. AYARLAR
# ------------------------------------------------------------------
DIZIN = "/content/drive/MyDrive/Colab Notebooks/datasets/rossman"
HAZIRLIK = f"{DIZIN}/hazirlik"
CIKTI = f"{DIZIN}/model"

EMB_BOYUT, GRU_BIRIM, COZUCU_BIRIM = 24, 128, 128
DROPOUT, L2 = 0.3, 1e-5
QUANTILES = [0.10, 0.50, 0.90]
SEEDLER = [42, 1337, 2024]

TAZE_PAY = 0.20        # cironun taze kategori payi
BIRIM_FIYAT = 8.0      # EUR / adet
SENARYOLAR = ["planli", "promo_yok", "promo_var"]

print("TF:", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

TF: 2.20.0 | GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
# ------------------------------------------------------------------
# 2. VERI + KALIBRASYON
# ------------------------------------------------------------------
meta = json.load(open(f"{HAZIRLIK}/meta.json"))
GECMIS, UFUK = meta["gecmis"], meta["ufuk"]
N_MAGAZA = meta["n_magaza"]
magaza_ort = np.array(meta["magaza_ort"], dtype=np.float32)
magaza_std = np.array(meta["magaza_std"], dtype=np.float32)
gelecek_kanal = meta["gelecek_kanal"]

IDX_ACIK  = gelecek_kanal.index("acik")
IDX_PROMO = gelecek_kanal.index("promo")
IDX_OKUL  = gelecek_kanal.index("okul")
IDX_TATIL = gelecek_kanal.index("tatil")

test = {k: v for k, v in np.load(f"{HAZIRLIK}/nn_test.npz").items()}
N_GECMIS_KANAL = test["X_gecmis"].shape[-1]
N_GELECEK_KANAL = test["X_gelecek"].shape[-1]
assert N_GELECEK_KANAL == len(gelecek_kanal), "Gelecek kanal uyusmazligi!"
assert test["X_gecmis"].shape[0] == N_MAGAZA, "Test satiri magaza sayisina esit degil!"

kal = json.load(open(f"{CIKTI}/kalibrasyon.json"))
K_ALT, K_UST = kal["k_alt"], kal["k_ust"]
print(f"kalibrasyon: {kal['secilen']} | k_alt {K_ALT} | k_ust {K_UST}")

sonuc = json.load(open(f"{CIKTI}/global_model_sonuc.json"))
smape_val  = {int(k): v for k, v in sonuc["val"]["magaza_smape"].items()}
smape_test = {int(k): v for k, v in sonuc["test"]["magaza_smape"].items()}

BASLANGIC = pd.Timestamp(meta["test_hedef"][0])
tarihler = pd.date_range(BASLANGIC, periods=UFUK, freq="D")
DEMO_BUGUN = str((BASLANGIC - pd.Timedelta(days=1)).date())   # origin gunu = "bugun"
print(f"demo bugun: {DEMO_BUGUN} | pencere {tarihler[0].date()} .. {tarihler[-1].date()}")

kalibrasyon: B_tek_k | k_alt 1.0511573553085327 | k_ust 1.0511573553085327
demo bugun: 2015-07-03 | pencere 2015-07-04 .. 2015-07-31


In [4]:
# ------------------------------------------------------------------
# 3. MAGAZA OZELLIKLERI (store.csv)  <-- canli demoda sorulacak
# ------------------------------------------------------------------
store = pd.read_csv(f"{DIZIN}/store.csv")
store = store.rename(columns={
    "Store": "magaza", "StoreType": "magaza_tipi", "Assortment": "urun_yelpazesi",
    "CompetitionDistance": "rakip_mesafe_m", "Promo2": "promo2"})
store = store[["magaza", "magaza_tipi", "urun_yelpazesi", "rakip_mesafe_m", "promo2"]]
store["rakip_mesafe_m"] = store["rakip_mesafe_m"].fillna(-1)   # -1 = bilinmiyor
print("magaza tipi dagilimi:", store.magaza_tipi.value_counts().to_dict())
print("urun yelpazesi     :", store.urun_yelpazesi.value_counts().to_dict())

magaza tipi dagilimi: {'a': 602, 'd': 348, 'c': 148, 'b': 17}
urun yelpazesi     : {'a': 593, 'c': 513, 'b': 9}


In [5]:
# ------------------------------------------------------------------
# 4. MODEL
# ------------------------------------------------------------------
def model_kur():
    g_in = keras.Input(shape=(GECMIS, N_GECMIS_KANAL), name="gecmis")
    f_in = keras.Input(shape=(UFUK, N_GELECEK_KANAL), name="gelecek")
    s_in = keras.Input(shape=(), dtype="int32", name="magaza")
    emb = layers.Embedding(N_MAGAZA, EMB_BOYUT,
                           embeddings_regularizer=keras.regularizers.l2(L2),
                           name="magaza_embedding")(s_in)
    emb = layers.Flatten(name="magaza_vektor")(emb)
    h = layers.GRU(GRU_BIRIM, return_sequences=True,
                   kernel_regularizer=keras.regularizers.l2(L2))(g_in)
    h = layers.Dropout(DROPOUT)(h)
    h = layers.GRU(GRU_BIRIM, kernel_regularizer=keras.regularizers.l2(L2))(h)
    h = layers.Dropout(DROPOUT)(h)
    baglam = layers.Concatenate()([h, emb])
    baglam = layers.Dense(COZUCU_BIRIM, activation="relu")(baglam)
    baglam_seq = layers.RepeatVector(UFUK)(baglam)
    d = layers.Concatenate()([f_in, baglam_seq])
    d = layers.GRU(COZUCU_BIRIM, return_sequences=True,
                   kernel_regularizer=keras.regularizers.l2(L2))(d)
    d = layers.Dropout(DROPOUT)(d)
    d = layers.TimeDistributed(layers.Dense(64, activation="relu"))(d)
    cikis = layers.TimeDistributed(layers.Dense(len(QUANTILES)), name="quantiles")(d)
    return keras.Model([g_in, f_in, s_in], cikis, name="rossmann_global")

modeller = []
for sd in SEEDLER:
    m = model_kur(); m.load_weights(f"{CIKTI}/global_seed{sd}.weights.h5"); modeller.append(m)
print(f"{len(modeller)} model yuklendi")

def ters_olcek(y_olcek, magaza_idx):
    return np.expm1(y_olcek * magaza_std[magaza_idx][:, None] + magaza_ort[magaza_idx][:, None])

def kfy(k):
    a = np.asarray(k, dtype=np.float32)
    return a[None, :] if a.ndim == 1 else a

def senaryo_girdisi(senaryo):
    X = test["X_gelecek"].copy()
    if senaryo == "promo_yok":
        X[:, :, IDX_PROMO] = 0.0
    elif senaryo == "promo_var":
        X[:, :, IDX_PROMO] = X[:, :, IDX_ACIK]    # promo sadece acik gunlerde
    return {"gecmis": test["X_gecmis"], "gelecek": X, "magaza": test["X_magaza"]}

3 model yuklendi


In [6]:
# ------------------------------------------------------------------
# 5. TAHMIN + KALIBRASYON + DONUSUM
# ------------------------------------------------------------------
acik = test["X_gelecek"][:, :, IDX_ACIK] > 0.5
magaza_no = test["magaza_no"].astype(int)
print(f"kapali gun orani: {100*(~acik).mean():.1f}%")

parcalar = []
for sen in SENARYOLAR:
    g = senaryo_girdisi(sen)
    p = np.mean([m.predict(g, batch_size=512, verbose=0) for m in modeller], axis=0)
    p10_o, p50_o, p90_o = p[..., 0], p[..., 1], p[..., 2]
    alt = p50_o - kfy(K_ALT) * (p50_o - p10_o)
    ust = p50_o + kfy(K_UST) * (p90_o - p50_o)

    p10 = np.maximum(ters_olcek(alt,   test["X_magaza"]), 0.0)
    p50 = np.maximum(ters_olcek(p50_o, test["X_magaza"]), 0.0)
    p90 = np.maximum(ters_olcek(ust,   test["X_magaza"]), 0.0)
    p10, p50, p90 = (np.where(acik, x, 0.0) for x in (p10, p50, p90))
    p10 = np.minimum(p10, p50); p90 = np.maximum(p90, p50)

    parcalar.append(pd.DataFrame({
        "magaza": np.repeat(magaza_no, UFUK),
        "tarih":  np.tile(tarihler.values, N_MAGAZA),
        "h":      np.tile(np.arange(1, UFUK + 1), N_MAGAZA),
        "senaryo": sen,
        "acik":  acik.ravel().astype(int),
        "promo": (g["gelecek"][:, :, IDX_PROMO] > 0.5).ravel().astype(int),
        "okul":  (test["X_gelecek"][:, :, IDX_OKUL] > 0.5).ravel().astype(int),
        "tatil": (test["X_gelecek"][:, :, IDX_TATIL] > 0.5).ravel().astype(int),
        "p10_eur": p10.ravel(), "p50_eur": p50.ravel(), "p90_eur": p90.ravel(),
    }))
    print(f"  {sen:<10} ort p50 {p50[acik].mean():8.1f} EUR | band {(p90-p10)[acik].mean():7.1f} EUR")

tahmin = pd.concat(parcalar, ignore_index=True)

# EUR -> ADET (donusum TEK YERDE)
for q in ["p10", "p50", "p90"]:
    tahmin[f"{q}_adet"] = tahmin[f"{q}_eur"] * TAZE_PAY / BIRIM_FIYAT

tahmin["gun_adi"] = tahmin["tarih"].dt.day_name()

# magaza ozellikleri + buyukluk referansi
idx_of = {int(mno): int(test["X_magaza"][i]) for i, mno in enumerate(magaza_no)}
tipik = {mno: float(np.expm1(magaza_ort[i])) for mno, i in idx_of.items()}
tahmin = tahmin.merge(store, on="magaza", how="left")
tahmin["tipik_gunluk_ciro_eur"] = tahmin["magaza"].map(tipik)
assert tahmin["magaza_tipi"].isna().sum() == 0, "store.csv eslesmesi eksik!"

# model kendi zayifligini raporlasin
tahmin["magaza_smape_val"]  = tahmin["magaza"].map(smape_val)
tahmin["magaza_smape_test"] = tahmin["magaza"].map(smape_test)
maske_t = test["maske"].astype(bool)
p_ham = np.mean([m.predict({"gecmis": test["X_gecmis"], "gelecek": test["X_gelecek"],
                            "magaza": test["X_magaza"]}, batch_size=512, verbose=0)
                 for m in modeller], axis=0)
a_ = p_ham[..., 1] - kfy(K_ALT) * (p_ham[..., 1] - p_ham[..., 0])
u_ = p_ham[..., 1] + kfy(K_UST) * (p_ham[..., 2] - p_ham[..., 1])
icinde = (test["y"] >= a_) & (test["y"] <= u_)
tahmin["magaza_kapsama_test"] = tahmin["magaza"].map(
    {int(mno): float(100.0 * icinde[i][maske_t[i]].mean())
     for i, mno in enumerate(magaza_no) if maske_t[i].sum() > 0})

# --- VAL metrikleri: KARAR ANINDA BILINEN (v3) ------------------------------
# Agent'in belirsizlik kapisi bunlari kullanir. Test metrikleri (yukarida)
# yalniz geriye donuk degerlendirme icindir; 07-03'te bilinemezler.
# Not: kalibrasyon katsayisi val'den ogrenildigi icin GLOBAL val kapsamasi
# tanim geregi ~%80'dir; magaza bazli dagilim ise bilgi tasir.
val_d = {k: v for k, v in np.load(f"{HAZIRLIK}/nn_val.npz").items()}
p_val = np.mean([m.predict({"gecmis": val_d["X_gecmis"], "gelecek": val_d["X_gelecek"],
                            "magaza": val_d["X_magaza"]}, batch_size=512, verbose=0)
                 for m in modeller], axis=0)
a_v = p_val[..., 1] - kfy(K_ALT) * (p_val[..., 1] - p_val[..., 0])
u_v = p_val[..., 1] + kfy(K_UST) * (p_val[..., 2] - p_val[..., 1])
icinde_v = (val_d["y"] >= a_v) & (val_d["y"] <= u_v)
maske_v = val_d["maske"].astype(bool)
no_v = val_d["magaza_no"].astype(int)
# 5 val origin'i 3 gun arayla ortusuyor; bagimsiz gozlem sayisi olarak
# FARKLI ACIK HEDEF GUNU sayilir (binom testinin n'i).
gun_v = val_d["origin_idx"][:, None] + np.arange(1, UFUK + 1)[None, :]
kap_val, n_val = {}, {}
for mno in np.unique(no_v):
    sec = no_v == mno
    mk = maske_v[sec]
    if mk.sum() == 0:
        continue
    kap_val[int(mno)] = float(100.0 * icinde_v[sec][mk].mean())
    n_val[int(mno)] = int(np.unique(gun_v[sec][mk]).size)
tahmin["magaza_kapsama_val"] = tahmin["magaza"].map(kap_val)
tahmin["magaza_n_val"] = tahmin["magaza"].map(n_val).astype("Int64")
print(f"val kapsama (kalibre): ort %{np.mean(list(kap_val.values())):.1f} | "
      f"n_val medyan {int(np.median(list(n_val.values())))} gun")

assert not any(c in tahmin.columns for c in ["gercek", "y", "y_ham", "Sales"]), "Sizinti!"


kapali gun orani: 14.0%


  planli     ort p50   6889.0 EUR | band  1696.6 EUR
  promo_yok  ort p50   6039.2 EUR | band  1555.6 EUR
  promo_var  ort p50   7816.8 EUR | band  1893.3 EUR
val kapsama (kalibre): ort %79.9 | n_val medyan 33 gun


In [7]:
# ------------------------------------------------------------------
# 6. DOGRULAMA
# ------------------------------------------------------------------
print("\n" + "="*58); print("DOGRULAMA"); print("="*58)
print(f"satir : {len(tahmin):,} (beklenen {N_MAGAZA*UFUK*len(SENARYOLAR):,}) | NaN {int(tahmin.isna().sum().sum())}")
print(f"monotonluk ihlali: {int((tahmin.p10_eur > tahmin.p50_eur).sum() + (tahmin.p50_eur > tahmin.p90_eur).sum())}")
print(f"kapali gun p50 sifir: {bool((tahmin.loc[tahmin.acik==0,'p50_eur']==0).all())}")

py = tahmin[tahmin.senaryo=="promo_yok"].p50_eur.sum()
pv = tahmin[tahmin.senaryo=="promo_var"].p50_eur.sum()
print(f"promo_var / promo_yok = {pv/py:.3f}   <- beklenen > 1")

print("\nmagaza tipine gore ort. gunluk p50 (planli, acik gunler):")
print(tahmin.query("senaryo=='planli' and acik==1")
      .groupby("magaza_tipi")
      .agg(ort_p50=("p50_eur","mean"), ort_smape=("magaza_smape_test","mean"),
           magaza=("magaza","nunique")).round(1).to_string())

print("\ncanli soru provasi - 07-10'da olceginin en ustunde calisacak 5 magaza:")
g = tahmin.query("senaryo=='planli' and tarih=='2015-07-10' and acik==1").copy()
g["oran"] = g.p50_eur / g.tipik_gunluk_ciro_eur
print(g.nlargest(5, "oran")[["magaza","magaza_tipi","p50_eur","tipik_gunluk_ciro_eur","oran"]]
      .round(2).to_string(index=False))


DOGRULAMA
satir : 93,660 (beklenen 93,660) | NaN 0
monotonluk ihlali: 0
kapali gun p50 sifir: True
promo_var / promo_yok = 1.294   <- beklenen > 1

magaza tipine gore ort. gunluk p50 (planli, acik gunler):
                  ort_p50  ort_smape  magaza
magaza_tipi                                 
a             6811.399902        8.4     602
b            10457.299805        8.5      17
c             6767.799805        8.5     148
d             6871.500000        8.1     348

canli soru provasi - 07-10'da olceginin en ustunde calisacak 5 magaza:
 magaza magaza_tipi      p50_eur  tipik_gunluk_ciro_eur  oran
    837           a  4476.750000                3114.28  1.44
    192           d 11363.759766                8080.78  1.41
    530           a  5794.169922                4138.68  1.40
    732           a  7280.200195                5249.74  1.39
    708           c  5579.700195                4078.46  1.37


/tmp/ipykernel_1393/2050222600.py:20: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  g = tahmin.query("senaryo=='planli' and tarih=='2015-07-10' and acik==1").copy()


In [8]:
# ------------------------------------------------------------------
# 7. KAYIT
# ------------------------------------------------------------------
yol = f"{CIKTI}/tahminler_tum_magazalar.parquet"
tahmin.to_parquet(yol, index=False)
print(f"\nkaydedildi -> {yol}  ({os.path.getsize(yol)/1e6:.1f} MB)")

tmeta = {
    "uretim_kaynagi": "global GRU + store embedding, 3 seed toplulugu",
    "demo_bugun": DEMO_BUGUN,
    "pencere": [str(tarihler[0].date()), str(tarihler[-1].date())],
    "takvim_kurali": f"'bugun' = {DEMO_BUGUN}; 'yarin' = {tarihler[0].date()}; "
                     f"pencere disi tarih sorulursa agent net hata dondurur",
    "ufuk": UFUK, "magaza_sayisi": int(N_MAGAZA), "senaryolar": SENARYOLAR,
    "kalibrasyon": {"yapi": kal["secilen"], "k_alt": K_ALT, "k_ust": K_UST,
                    "test_kapsama_sonra": kal["test"]["kapsama_sonra"]},
    "adet_donusumu": {"taze_pay": TAZE_PAY, "birim_fiyat_eur": BIRIM_FIYAT,
                      "formul": "adet = eur * taze_pay / birim_fiyat"},
    "kolonlar": list(tahmin.columns),
    "uyari": "promo_var senaryosu kars-olgusal: 28 gun kesintisiz promo egitim dagiliminda yok, "
             "mutlak degil GORECE karsilastirma icin kullanilmali",
    "sizinti_notu": "gercek satis kolonu bilincli olarak YOK",
    "belirsizlik_kapisi": "magaza_smape_val + magaza_kapsama_val (binom, n=magaza_n_val); "
                          "magaza_*_test kolonlari yalniz geriye donuk degerlendirme icindir",
}
with open(f"{CIKTI}/tahmin_meta.json", "w") as f:
    json.dump(tmeta, f, indent=2)
print("tahmin_meta.json kaydedildi")
print("\n-> Iki dosyayi indir, yerel Rossman/ altina koy.")


kaydedildi -> /content/drive/MyDrive/Colab Notebooks/datasets/rossman/model/tahminler_tum_magazalar.parquet  (2.9 MB)
tahmin_meta.json kaydedildi

-> Iki dosyayi indir, yerel Rossman/ altina koy.
